# EELS Analysis with *HyperSpy* and *eXSpy*

Tutorial for the **DPG Spring Meeting of the Condensed Matter Section 2026**

> Dresden, 8th March 2026

In this hands-on session we walk through a complete EELS analysis pipeline — from raw spectrum alignment to quantitative curve fitting to statistical denoising. We use two real datasets:

- **Low-loss map:** a STEM-EELS spectrum image of a plasmonic aluminum nanotriangle, rich in localised surface plasmon resonance (LSPR) modes.
- **Core-loss spectra:** hexagonal boron nitride (h-BN) spectra from the [EELS Data Base](https://eelsdb.eu/), a prototypical 2D material with two well-separated ionisation edges (B-K, N-K).
- - **Mystery core-loss map to play with** provided by Benedikt Haas and Mairi Mccauley.

**Table of Contents:**

1. [Setup and data loading](#1.-Setup-and-data-loading)
2. [Spectral alignment and drift correction](#2.-Spectral-alignment-and-drift-correction)
3. [Low-loss analysis — thickness and elastic scattering, ZLP removal](#3.-Low-loss-analysis)
4. [Core-loss quantification by curve fitting](#4.-Core-loss-quantification-by-curve-fitting)
5. [Fine structure modelling](#5.-Fine-structure-modelling)
6. [Fitting across a full spectrum image with multifit](#6.-Fitting-across-a-full-spectrum-image-with-multifit)
7. [PCA denoising in low-signal regimes](#7.-PCA-denoising-in-low-signal-regimes)
8. [Blind source separation of plasmon modes](#8.-Blind-source-separation-of-plasmon-modes)


---
## 1. Setup and data loading

Since HyperSpy 2.0 the EELS-specific functionality lives in a separate extension package called **eXSpy**. You import HyperSpy for the core machinery (loading files, model fitting, decomposition, plotting) and eXSpy to unlock the `EELSSpectrum` signal type with all its specialised methods. Importing eXSpy is enough, it registers itself automatically.


In [ ]:
# Interactive plotting: pick the backend that matches your environment
#%matplotlib widget
%matplotlib qt5
#%matplotlib inline

In [ ]:
import hyperspy.api as hs
import exspy
import matplotlib.pyplot as plt
import numpy as np

# Lay out navigator + signal windows side by side
hs.preferences.Plot.widget_plot_style = 'horizontal'


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

### Fetch h-BN core-loss and low-loss reference spectra

For the core-loss fitting sections we can use published h-BN spectra downloaded directly from [eelsdb.eu](https://eelsdb.eu/). The `exspy.data.eelsdb()` function queries the database and returns `EELSSpectrum` objects with metadata already populated (beam energy, angles, elements). To speed things up we have also downloaded the spectra to the **data** folder. 

> Note that this step requires an internet connection if downloading directly from the database.


In [ ]:
# Core-loss spectrum containing the B-K and N-K edges from the database
cl = exspy.data.eelsdb(
    title='Hexagonal Boron Nitride',
    spectrum_type='coreloss'
)[0]

# if no internet connection use this: 
#cl = hs.load('data/hBN_coreloss.hspy')
#print(cl.data.min(), cl.data.max())
#print(cl.axes_manager[-1])


cl.plot()

In [ ]:
# Corresponding low-loss spectrum (needed for plural-scattering correction)
#ll = exspy.data.eelsdb(
#    title='Hexagonal Boron Nitride',
#    spectrum_type='lowloss'
#)[0]

ll = hs.load('data/hBN_lowloss.hspy')
ll.plot()

The core-loss spectrum shows two prominent edges: Boron K near 188 eV and Nitrogen K near 401 eV. The low-loss is dominated by the zero-loss peak.


In [ ]:
# you can also then select specific energy loss ranges to focus on
b_edge = cl.isig[180.:220.]
b_edge.plot()

### Load the aluminum nanotriangle low-loss spectrum image

This is a STEM-EELS map acquired over a lithographically fabricated aluminum nanotriangle. The low-loss region (0 eV to ~30 eV) contains the zero-loss peak and several localised surface plasmon resonance modes whose spatial distribution maps directly onto the triangle geometry. We will use this dataset for alignment, low-loss analysis, and blind source separation.


In [ ]:
#  Load the Al triangle low-loss spectrum image (.dm3) 
# Adjust the path to point to your file
al = hs.load('data/Al_triangle_lowloss.hspy')

In [ ]:
# Quick look at shape, calibration, and units
al.axes_manager

In [ ]:
# Plot — a navigator image and the spectrum at the active pixel
al.plot()

Explore the dataset by dragging the red cursor across the navigator image. Notice how the plasmon peaks shift in position and intensity as you move from the triangle edges to the corners to the surrounding vacuum.


### Experimental parameters and sample composition

Quantitative EELS needs three numbers: beam energy and the two semi-angles describing the illumination and collection geometry. Data from the EELS database comes pre-filled, but with your own data you must set them explicitly:


In [ ]:
# ─────── Al triangle parameters ────────────────────────────────
# Check what the file brought in
print(al.metadata)

In [ ]:
al.metadata.General.title = 'Al bowtie'

In [ ]:
# Set or verify microscope parameters for the Al data
# Adjust these to match your actual acquisition conditions!
al.set_microscope_parameters(
    beam_energy=60,            # keV
    convergence_angle=10,       # mrad  (semi-angle)
    collection_angle=20         # mrad  (semi-angle)
)

In [ ]:
# ── h-BN parameters (from the database) ─────────────────────────────
cl.set_microscope_parameters(
    beam_energy=300,            # keV
    convergence_angle=0.2,      # mrad
    collection_angle=2.55       # mrad
)

# Declare which elements are present: this tells the model builder which ionisation edges to include
cl.add_elements(('B', 'N'))
print('Elements registered:', cl.metadata.Sample.elements)


### Identifying edges in an unknown spectrum

`print_edges_near_energy()` prints a table of candidate edges within a window around a given energy loss. `edges_at_energy()` gives you an interactive GUI: drag across a region of the spectrum and toggle edge labels on and off:


In [ ]:
# What edges are near 188 eV? (B-K is at 188 eV)
cl.edges_at_energy(energy=188)

In [ ]:
cl.plot()

In [ ]:
# Interactive edge identification (opens a widget)
cl.edges_at_energy()

---
## 2. Spectral alignment and drift correction

Instabilities in the spectrometer or specimen drift during acquisition shift the zero-loss peak away from 0 eV. Correcting this is the first preprocessing step in any EELS workflow.

`align_zero_loss_peak()` estimates the ZLP position (assuming it is the most intense feature) and shifts each spectrum so the ZLP sits at exactly 0 eV. A two-pass strategy is robust:

1. A broad search window with **pixel-precision** alignment catches large offsets.
2. A narrow window with **sub-pixel interpolation** refines the result.

We apply this to the aluminum spectrum image:


In [ ]:
# Pass 1: coarse, pixel-level alignment
al.align_zero_loss_peak(
    signal_range=(0., 5.),
    subpixel=False,
    show_progressbar=True,
)

# Pass 2: sub-pixel refinement over a tight window
al.align_zero_loss_peak(
    signal_range=(-0.5, 0.5),
    subpixel=True,
    show_progressbar=True,
)


In [ ]:
# Verify: the ZLP should now be centred at 0 eV
al.plot()

> **Also useful:** The `also_align` argument lets you apply the same shifts to a simultaneously acquired core-loss spectrum:
> ```python
> al.align_zero_loss_peak(signal_range=(-0.5, 0.5), subpixel=True,
>                         also_align=[core_loss_signal])
> ```


In [ ]:
al.save('data/Al_triangle_lowloss_aligned_mine.hspy')

---
## 3. Low-loss analysis

The low-loss region carries information about specimen thickness and the dielectric response. On the aluminum nanotriangle we can extract:

- **Elastic scattering intensity** (I₀): the ZLP integral, separated from the inelastic tail by a threshold.
- **Relative thickness** t/λ via the log-ratio of total to elastic intensity. With density known (~2.7 g/cm³ for Al), this becomes an absolute thickness map.


**Below we are setting our threshold but this can also be estimated** by using:


threshold = al.estimate_elastic_scattering_threshold()

i0 = al.estimate_elastic_scattering_intensity(threshold=threshold)


In [ ]:
# Estimate the elastic scattering intensity (ZLP integral)
# We use a fixed threshold of 0.1 eV to separate elastic from inelastic scattering. 
# This value should sit just beyond the ZLP tail. 
#This is high energy resolution data so this number is low, a reasonable range could also be as high as 3eV. 

i0 = al.estimate_elastic_scattering_intensity(threshold=0.1)

In [ ]:
i0.plot()
plt.title('Elastic scattering intensity I_0')

In [ ]:
# Relative thickness map via the log-ratio method
t_over_lambda = al.estimate_thickness(threshold=0.1)
t_over_lambda.plot()
plt.title('Relative thickness t/lambda')

The thickness map should trace the triangle shape: thicker in the centre, tapering to the edges and zero in the surrounding vacuum.


### Zero-loss peak removal methods

The ZLP obscures the low-loss fine structure and must be removed for dielectric analysis or clean plasmon mapping. eXSpy provides several 
approaches:

In [ ]:
# Work on a copy so we don't modify the aligned data
al_zlp = al.deepcopy()

What to do if you have no decent ZLP reference or it is not working? 
If your zero-loss peak contains phonon or other low-energy features that interfere with deconvolution, you can construct a clean ZLP reference in two ways:

- Mirror the negative tail: The negative energy-loss side is free of inelastic features, so flipping it onto the positive side produces a symmetric ZLP suitable for deconvolution.
- Fit and extend beyond a cutoff: Fit an exponential decay to the ZLP tail just before the phonon onset, then replace everything beyond that cutoff with the fitted curve. This preserves the natural peak shape while removing the phonon contribution.

In [ ]:
# mirroring the negative energy-loss tail 
zlp_ref = al_zlp.inav[3, 3]
zlp_clean = zlp_ref.deepcopy()

energy = zlp_clean.axes_manager[-1].axis
i_zero = np.argmin(np.abs(energy))

n_neg = i_zero
n_pos = len(energy) - i_zero - 1
n = min(n_neg, n_pos)

# Get the negative side (excluding zero), flip it
neg_tail = zlp_clean.data[i_zero - n : i_zero]
neg_tail_flipped = neg_tail[::-1].copy()

# Replace positive side with mirrored negative
zlp_clean.data[i_zero + 1 : i_zero + 1 + n] = neg_tail_flipped

# Zero out anything beyond
if n < n_pos:
    zlp_clean.data[i_zero + 1 + n:] = 0

# Compare
zlp_ref.isig[-0.5:2.].plot()
zlp_clean.isig[-0.5:2.].plot()

In [ ]:
# or expanding the ZLP peak past a cut off 

zlp_ref = al_zlp.inav[3, 3]
zlp_clean = zlp_ref.deepcopy()
energy = zlp_clean.axes_manager[-1].axis

# Fit a power-law or exponential to a small window on the positive tail
# just before the phonon features start
fit_region = (energy > 0.03) & (energy < 0.06)

# Exponential fit: I = A * exp(-E/tau)
from scipy.optimize import curve_fit

def exp_decay(x, A, tau):
    return A * np.exp(-x / tau)

popt, _ = curve_fit(exp_decay, energy[fit_region], zlp_ref.data[fit_region],
                    p0=[zlp_ref.data[fit_region][0], 0.02])

print(f'A: {popt[0]:.0f}, tau: {popt[1]:.4f} eV')

# Replace everything from 0.05 eV onwards with the fitted tail
extend_mask = energy >= 0.05
zlp_clean.data[extend_mask] = exp_decay(energy[extend_mask], *popt)

# Compare
fig, ax = plt.subplots()
ax.semilogy(energy, zlp_ref.data, 'k-', alpha=0.5, label='Original')
ax.semilogy(energy, zlp_clean.data, 'r-', label='Extended tail')
ax.set_xlim(-0.1, 1.0)
ax.set_xlabel('Energy loss (eV)')
ax.legend()
ax.set_title('ZLP with fitted tail extension')
plt.show()

In [ ]:
# Method 1: Fourier-log deconvolution demo on a single spectrum
s_single = al_zlp.inav[19, 6]
#zlp_ref = al_zlp.inav[0, 0] if you have a vacuum region in your map, you can skip the previous step and use this as your reference spectra

ssd_flog = s_single.fourier_log_deconvolution(zlp_clean)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_single.axes_manager[-1].axis, s_single.data, 'k-', alpha=0.4, label='Original')
ax.plot(ssd_flog.axes_manager[-1].axis, ssd_flog.data, 'r-', label='Fourier-log SSD')
ax.set_xlim(1.5, 9)
ax.set_ylim(-500, 5000)  # adjust to see the plasmon peaks
ax.set_xlabel('Energy loss (eV)')
ax.set_ylabel('Intensity')
ax.legend()
ax.set_title('Fourier-log deconvolution — plasmon region')
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: Richardson-Lucy deconvolution (iterative, needs a PSF):

zlp_for_deconv = zlp_clean.deepcopy()

s_single = al_zlp.inav[50, 50] # adjust to pixel in ROI

# Normalize
scale = s_single.isig[-0.3:0.3].integrate1D(-1).data / zlp_for_deconv.isig[-0.3:0.3].integrate1D(-1).data
zlp_scaled = zlp_for_deconv * scale

# Now do the RL
ssd_rl = s_single.richardson_lucy_deconvolution(zlp_scaled, iterations=10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_single.axes_manager[-1].axis, s_single.data, 'k-', alpha=0.3, label='Original')
ax.plot(ssd_rl.axes_manager[-1].axis, ssd_rl.data, 'b-', label='Richardson-Lucy (10 iter.)')
ax.set_xlim(1.5, 9)
ax.set_xlabel('Energy loss (eV)')
ax.set_ylabel('Intensity')
ax.legend()
ax.set_title('ZLP removal comparison — on structure')
plt.tight_layout()
plt.show()

In [ ]:
#Compare Fourier Log and RL using the same pixel for both methods
s_single = al_zlp.inav[50, 50]

# Fourier-log
ssd_flog = s_single.fourier_log_deconvolution(zlp_clean)

# Richardson-Lucy (normalize ZLP first)
zlp_for_deconv = zlp_clean.deepcopy()
scale = s_single.isig[-0.3:0.3].integrate1D(-1).data / zlp_for_deconv.isig[-0.3:0.3].integrate1D(-1).data
zlp_scaled = zlp_for_deconv * scale
ssd_rl = s_single.richardson_lucy_deconvolution(zlp_scaled, iterations=10)

# Compare
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ssd_flog.axes_manager[-1].axis, ssd_flog.data,
        label='Fourier-log')
ax.plot(ssd_rl.axes_manager[-1].axis, ssd_rl.data,
        label='Richardson-Lucy (10 iter.)')
ax.set_xlim(0.5, 9)
ax.set_xlabel('Energy loss (eV)')
ax.set_ylabel('Intensity')
ax.set_title('ZLP removal comparison — same pixel')
ax.legend()
plt.tight_layout()
plt.show()

Fourier-ratio deconvolution (for core-loss, needs low-loss reference):

In [ ]:
# This is typically used on core-loss data with a low-loss reference
# ssd_frat = core_loss.fourier_ratio_deconvolution(low_loss)

**Fourier-log** deconvolution recovers the single scattering distribution and removes both the ZLP and plural scattering contributions. 
**Fourier-ratio** uses a measured ZLP reference (e.g. from vacuum) and is useful when the ZLP shape varies across the dataset.
For the decomposition in Sections 7–8, simple **cropping** past the ZLP tail is sufficient since we only care about plasmon peak positions and 
spatial distributions.

---
## 4. Core-loss quantification by curve fitting

Now we switch to the h-BN core-loss spectrum. The idea is to build a physical model — a power-law background plus theoretical cross-section profiles for each ionisation edge — and fit it to the data. The resulting edge intensities are proportional to atomic concentrations.

### Building the model

Because we already declared elements and microscope parameters, `create_model()` sets everything up automatically: a `PowerLaw` background and one `EELSCLEdge` component per element.

Passing the low-loss spectrum tells HyperSpy to **convolve the model with the low-loss** before comparing to the data. This accounts for plural scattering and is important whenever your specimen is not extremely thin:


In [ ]:
# Build the EELS model with plural-scattering correction
m = cl.create_model(low_loss=ll)

In [ ]:
# Let's check what got created automatically
m.components

Each `EELSCLEdge` uses tabulated generalised oscillator strengths (GOS) to calculate the edge shape. By default eXSpy downloads a freely available dataset in [gosh format](https://gitlab.com/gguzzina/gosh) from Zenodo. Alternatives include the relativistic Dirac GOS (`create_model(low_loss=ll, GOS='Dirac')`) and the Hartree-Slater tables which is coming with Gatan DigitalMicrograph.


### Fitting with `smart_fit`

A plain least-squares fit can struggle with EELS because background and edges are strongly correlated. `smart_fit()` handles this by fitting iteratively: background first, then one edge at a time from high to low energy loss, then everything together. This sequential approach converges much more reliably:


In [ ]:
m.enable_fine_structure() #try to disable this and see what your fit looks like
m.smart_fit()

In [ ]:
m.plot(plot_components=True)


### Reading off the composition

`quantify()` reports the fitted edge intensities. For h-BN we expect a B/N ratio close to 1:


In [ ]:
m.quantify()

---
## 5. Fine structure modelling

The near-edge fine structure (ELNES) encodes information about bonding and local electronic structure that the tabulated GOS profiles cannot capture. eXSpy offers two approaches:

1. **Spline interpolation**: a smooth curve replaces the theoretical profile over a defined energy window.
2. **Explicit peak functions**:  Gaussians (or other components) represent individual white lines, giving measurable positions, widths and areas.

### Approach 1 — spline fine structure

Two parameters control the spline:
- `fine_structure_width`: how many eV from the onset are modelled (default 30).
- `fine_structure_smoothing`: 0 to 1 (default 0.3); lower = smoother.


In [ ]:
# Turn on spline fine structure for all edges
m.enable_fine_structure()

# Refine onset positions and widths to better match the data
m.set_signal_range(160.)
m.components.B_K.onset_energy.value = 194
m.components.N_K.onset_energy.value = 402.5
m.components.B_K.fine_structure_width = 40
m.components.N_K.fine_structure_width = 45
m.components.B_K.fine_structure_smoothing = 0.4

m.smart_fit()
m.plot(plot_components=True)


In [ ]:
# B/N ratio should now be closer to 1
m.quantify()

Getting the fine structure right matters, especially when deconvolving plural scattering, small errors in the near-edge region propagate into the intensity ratio.


### Approach 2 - Gaussians for white-line analysis

To *measure* individual fine-structure peaks (e.g. the B-K π\* and σ\* lines), replace part of the spline with Gaussian components. Register them as belonging to a particular edge via `fine_structure_components`, and the spline covers only the remainder of the window:


In [ ]:
# Two Gaussians for the first two B-K white lines
g_pi = hs.model.components1D.GaussianHF(centre=194.7, fwhm=3., height=5)
g_pi.name = 'B_K_pi'

g_sigma = hs.model.components1D.GaussianHF(centre=201.9, fwhm=5., height=5)
g_sigma.name = 'B_K_sigma'

# Register them as fine-structure components of the B-K edge
m.components.B_K.fine_structure_components.update((g_pi, g_sigma))

# Spline handles only the region beyond ~205 eV
m.components.B_K.fine_structure_spline_active = True
m.components.B_K.fine_structure_spline_onset = (
    205. - m.components.B_K.onset_energy.value
)

m.smart_fit()
m.plot(plot_components=True)

In [ ]:
# Read off the white-line parameters
print(f'pi* peak:    centre = {g_pi.centre.value:.1f} eV,  '
      f'FWHM = {g_pi.fwhm.value:.1f} eV,  height = {g_pi.height.value:.2f}')
print(f'sigma* peak: centre = {g_sigma.centre.value:.1f} eV,  '
      f'FWHM = {g_sigma.fwhm.value:.1f} eV,  height = {g_sigma.height.value:.2f}')

---
## 6. Fitting across a full spectrum image with multifit

Everything so far operated on a single h-BN spectrum. The real payoff comes when you apply the same model to every pixel in a spectrum image. `multifit()` loops over all navigation positions, using the previous pixel's result as starting values for the next.

The EELS-specific variant `multifit(kind='smart')` applies the same sequential edge-by-edge strategy as `smart_fit()` at every pixel.

**Practical workflow:**
1. Fit one good spectrum to get the starting parameters right.
2. Send those values everywhere with `assign_current_values_to_all()`.
3. Run `multifit(kind='smart')`.
4. Extract fitted parameters as maps or linescans.


In [ ]:
#  Load the core-loss map 
# Adjust the path to point to your file
cl_map = hs.load('data/coreloss_map1_aligned_binned.hspy')
haadf = np.load('data/HADFmap1.npy')

In [ ]:
# Set or verify microscope parameters for the Al data
# Adjust these to match your actual acquisition conditions!
cl_map.set_microscope_parameters(
    beam_energy=60,            # keV
    convergence_angle=10,       # mrad  (semi-angle)
    collection_angle=20         # mrad  (semi-angle)
)

In [ ]:
cl_map.plot()

In [ ]:
cl_map.edges_at_energy()

In [ ]:
# Crop to core-loss region only (adjust the start energy to be 
# ~30 eV before your first edge for good background fitting)
cl_core = cl_map.isig[310.:]  # start well before Ca L at ~346 eV

cl_core.add_elements(['C', 'Ca', 'O', 'Au'])

# Recreate the model from scratch
m_map = cl_core.create_model(auto_background=True)

m_map.enable_fine_structure(edges_list=['Ca_L3'])

In [ ]:
# Check what edges were created
print(m_map.components)

In [ ]:
# Fit a single spectrum first
cl_core.axes_manager.indices = (10, 20)
m_map.set_all_edges_intensities_positive()

m_map.smart_fit()
m_map.plot(plot_components=True)

In [ ]:
# Step 2–3: broadcast and fit all spectra
m_map.assign_current_values_to_all()
m_map.multifit(kind='smart', show_progressbar=True)


In [ ]:
Ca_int = m_map.components.Ca_L3.intensity.as_signal()

plt.figure()
plt.imshow(Ca_int.data, cmap='viridis', vmin=0, vmax=2.5)
plt.colorbar(label='Ca L$_{2,3}$ intensity (a.u.)')
plt.title('Ca L$_{2,3}$ elemental map')
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14, 4))

axes[0].imshow(haadf.data, cmap='gray')
axes[0].set_title('HAADF')

axes[1].imshow(Ca_int.data, cmap='viridis', vmin=0)
axes[1].set_title('Ca L$_{2,3}$')

plt.tight_layout()
plt.show()

With a 2D spectrum image the same `as_signal()` call produces 2D elemental maps that you can plot with any matplotlib colourmap.


---
## 7. PCA denoising in low-signal regimes

High spatial resolution or short dwell times produce noisy spectra. PCA offers an effective denoising strategy: decompose the dataset, keep only the components that carry real signal.

Two things to keep in mind for EELS:
- **Poissonian noise normalisation** is essential. Electron counts follow Poisson statistics; set `normalize_poissonian_noise=True`.
- The **scree plot** tells you how many components to keep, look for the elbow where the curve flattens.
  
I will demonstrate this on one of our aluminum bowtie low-loss maps, which
has moderate noise from the short pixel dwell time used during acquisition.


NOte: Before decomposition, the ZLP had already been aligned here. 


In [ ]:
# Load the first aluminum bowtie low-loss spectrum image
al_pca = hs.load('data/Al_triangle_lowloss.hspy')
print(al_pca)

al_pca.plot()

In [ ]:
# Crop to the plasmon region — remove the ZLP
al_pca_crop = al_pca.isig[0.5:8]

In [ ]:
# Quick look at the mean spectrum after cropping
al_pca_crop.mean().plot()
plt.title('Mean low-loss spectrum (plasmon region)')


### Decomposition and scree plot


In [ ]:
al_pca.decomposition(normalize_poissonian_noise=True, algorithm='SVD')

In [ ]:
al_pca.plot_explained_variance_ratio(n=150)

### Inspecting factors and loadings

Factors are spectral shapes; loadings are their spatial weights. The first few components should capture the dominant plasmon peaks and their spatial
distributions on the bowtie structure. Later components will be noise.


In [ ]:
al_pca.plot_decomposition_factors(comp_ids=6)
al_pca.plot_decomposition_loadings(comp_ids=6)

### Reconstruct the denoised signal

Choose the number of components from the scree plot elbow. Reconstruct the signal using only those components — the noise is left behind.


In [ ]:
n_components = 30  # adjust based on your scree plot
al_denoised = al_pca.get_decomposition_model(n_components)

In [ ]:
# Side-by-side comparison at a single pixel
# Pick a pixel on the bowtie structure (adjust indices to your data)
ix, iy = 30, 30

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    al_pca.axes_manager[-1].axis,
    al_pca.inav[ix, iy].data,
    color='grey', alpha=0.6, label='Original (noisy)'
)
ax.plot(
    al_denoised.axes_manager[-1].axis,
    al_denoised.inav[ix, iy].data,
    color='tab:red', lw=1.5, label=f'PCA denoised ({n_components} comp.)'
)
ax.set_xlabel('Energy loss (eV)')
ax.set_ylabel('Intensity (counts)')
ax.set_title('PCA denoising: single pixel comparison')
ax.legend()
plt.tight_layout()
plt.show()

### Residual check

Always inspect the residual (original − model). It should look like featureless noise. Structured patterns mean real signal was discarded:


In [ ]:
residual = al_pca - al_denoised

# Compare mean spectra: the residual should be flat
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(residual.axes_manager[-1].axis, residual.mean().data)
ax.axhline(0, color='k', ls='--', lw=0.5)
ax.set_xlabel('Energy loss (eV)')
ax.set_ylabel('Residual intensity')
ax.set_title('Mean residual — should be featureless')
plt.tight_layout()
plt.show()

---
## 8. Blind source separation of plasmon modes

We return to the aluminum nanotriangle low-loss data for the final section. Plasmonic nanostructures support several distinct resonance modes whose spectra and spatial distributions overlap in the raw data. Blind source separation lets us disentangle these modes purely from the statistics of the dataset. No peak shape assumptions required.

We will use two complementary algorithms:

- **SVD** for an initial scree-plot assessment of how many components matter.
- **Non-negative Matrix Factorisation (NMF)** for a physically interpretable decomposition. Since EELS counts are non-negative, forcing the factors and loadings to be non-negative often yields components that map directly onto physical plasmon modes.

The dataset $\mathbf{X}$ over spatial coordinates $\mathbf{R}=(x,y)$ and energy loss $E$ is factorised as:

$$\mathbf{X}_{\mathbf{R} \times E} = \mathbf{A}_{\mathbf{R} \times k}\, \mathbf{B}^T_{k \times E}$$

Each factor–loading pair ($A_k$, $B_k$) represents one source: a spatial map showing *where* that mode is excited and a spectrum showing its energy-loss signature.


### Preprocessing

Before running NMF we need to ensure the data is properly prepared:
- The ZLP alignment was done in Section 2.
- We crop to the energy-loss range of interest, removing the zero-loss peak.
- We remove any spikes from the dataset.
- We verify all values are non-negative (required by NMF).


In [ ]:
# Work on a copy so we keep the original intact
al_bss = al.deepcopy()

In [ ]:
# Remove spikes (cosmic rays, X-ray hits on the detector)
al_bss.spikes_removal_tool(interactive=False)

In [ ]:
# Crop to the plasmon region — adjust the range to your data
# Typically you want to start just above the ZLP tail
al_bss = al_bss.isig[0.1:8.]  # eV, removes the ZLP

In [ ]:
# Ensure all values check
al_bss.data = np.maximum(al_bss.data, 0)

### SVD and scree plot

First we assess how many significant components the dataset contains:


In [ ]:
al_bss.decomposition(normalize_poissonian_noise=True, algorithm='SVD')
al_bss.plot_explained_variance_ratio(n=40)

Count the components above the noise. For a triangular nanoparticle you typically expect a handful of dipolar and multipolar modes plus a bulk plasmon contribution, somewhere in the range of 4–8 significant components. Here more complex structure= more components needed.


### NMF decomposition

Choose the number of components based on the scree plot. It is worth trying a few values. If $k$ is too large, NMF tends to split genuine modes into asymmetric halves. Increase from a low number and watch for the appearance of new physically meaningful maps:


In [ ]:
# Set output_dimension based on your scree plot analysis
n_nmf = 15  # ← adjust this!

al_bss.decomposition(
    normalize_poissonian_noise=True,
    algorithm='NMF',
    output_dimension=n_nmf
)

### Visualising the plasmon modes

Each factor–loading pair should correspond to a distinct plasmon resonance. The loadings (spatial maps) show where each mode is excited
on the bowtie, look for:

- **Corner/tip modes** lighting up at the triangle vertices
- **Gap modes** concentrated in the junction between the two triangles
- **Edge modes** running along the sides
- **Bulk/volume plasmon** spread across the interior


The corresponding factors (spectra) show their energy-loss positions.


In [ ]:
# Maps only, one per mode
# al_bss.plot_decomposition_loadings()
al_bss.plot_decomposition_loadings(comp_ids=range(6), figsize=(12, 8))

# Plot only specific components
# al_bss.plot_decomposition_loadings(comp_ids=[2, 3, 4, 5,6,7])

In [ ]:
# Spectra only
al_bss.plot_decomposition_factors()

In [ ]:
factors = al_bss.get_decomposition_factors()
loadings = al_bss.get_decomposition_loadings()

fig, axes = plt.subplots(2, n_nmf, figsize=(3 * n_nmf, 6))

for i in range(n_nmf):
    # Spatial map (loading)
    axes[0, i].imshow(loadings.inav[i].data, cmap='viridis')
    axes[0, i].set_title(f'Mode {i}')
    axes[0, i].axis('off')

    # Spectrum (factor)
    axes[1, i].plot(factors.axes_manager[-1].axis, factors.inav[i].data)
    axes[1, i].set_xlabel('eV')
    if i == 0:
        axes[1, i].set_ylabel('Intensity')

plt.suptitle('NMF decomposition — Al bowtie plasmon modes', y=1.02)
plt.tight_layout()
plt.show()


### Denoising the full spectrum image via SVD

We can also use the SVD to reconstruct a denoised version of the complete spectrum image, keeping only the signal components:


In [ ]:
# Re-run SVD (NMF overwrites the stored decomposition)
al_bss.decomposition(normalize_poissonian_noise=True, algorithm='SVD')

al_denoised = al_bss.get_decomposition_model(n_nmf)
al_denoised.plot()

## ICA (alternative)

Independent component analysis offers another factorisation — it seeks statistically independent components rather than non-negative or orthogonal ones:

```python
al_bss.decomposition(normalize_poissonian_noise=True, algorithm='SVD')
al_bss.blind_source_separation(number_of_components=12)

al_bss.plot_bss_results()
al_bss.plot_bss_loadings()
al_bss.plot_bss_factors()
```


## What to remember:

We have gone from raw spectra to quantitative results in a single notebook:

- **Alignment** via `align_zero_loss_peak()` removes spectral drift across the aluminum spectrum image.
- **Low-loss analysis** produces thickness maps of the nanotriangle.
- **Model-based quantification** with `create_model()` + `smart_fit()` + `quantify()` extracts elemental concentrations from h-BN core-loss edges, with plural-scattering correction through the low-loss.
- **Fine structure** can be modelled with splines or explicit Gaussian components for white-line ratio analysis.
- **`multifit(kind='smart')`** automates fitting across entire spectrum images, producing elemental maps.
- **PCA denoising** via `decomposition()` + `get_decomposition_model()` recovers signal from noisy data.
- **NMF** on the aluminum nanotriangle separates overlapping plasmon resonance modes without assuming peak shapes.

All of these steps produce HyperSpy signal objects that you can save, plot, export, and combine — building reproducible analysis pipelines that you can
adapt directly to your own data.


**Further reading:**
- [eXSpy EELS user guide](https://exspy.readthedocs.io/en/v0.2.1/user_guide/eels.html)
- [HyperSpy decomposition docs](https://hyperspy.org/hyperspy-doc/current/user_guide/mva/decomposition.html)
- [exspy-demos on GitHub](https://github.com/hyperspy/exspy-demos)
- R.F. Egerton, *Electron Energy-Loss Spectroscopy in the Electron Microscope*, Springer, 2011
